# Chapter 13. 외부 데이터로 분석을 확장하기

> **수업 과제 | GitHub: `cyw0927` | 작성일: 2026-10-14**

이 노트북은 **실제 네트워크 API 요청 0회**를 기본으로 합니다. 인가되지 않은 외부 수집을 실행하지 않고, 공식 출처·현재 이용조건·Network Gate·Secret 상태·Snapshot/Metadata 계약·검증·병합·해석 한계를 재현합니다.

**완료 상태 구분**
- 실제 외부 API 수집: **SKIPPED** (키/실행 승인 미확인)
- 실제 외부 데이터 품질과 병합: **NO_DATA**
- 별도 병합 규칙 실습: **SYNTHETIC_DEMO_ONLY**
- 내부 월별 주문 금액: **실제 저장소 processed 데이터 기반 집계**

핵심: **수집할 수 있다 ≠ 수집해도 된다 / HTTP 200 ≠ 올바른 데이터 / 함께 움직인다 ≠ 원인이다.**

## 0. 환경과 프로젝트 루트

Notebook은 어느 하위 폴더에서 실행하더라도 프로젝트 루트를 찾아야 합니다. `src`와 `scripts`는 수업 공식 공개 저장소의 Chapter 13 공통 구현을 사용합니다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "requirements.txt").exists() and (path / "src").is_dir():
            return path
    raise FileNotFoundError("저장소 루트를 찾을 수 없습니다.")

ROOT = find_project_root(Path.cwd())
REPORT_DIR = ROOT / "reports"
IMAGE_DIR = ROOT / "assignments" / "chapter13" / "images"
PROCESSED_DIR = ROOT / "data" / "processed"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Python:", sys.executable)
print("Repo root:", ROOT)
print("Input directory:", PROCESSED_DIR)
print("No live API collection in this notebook")

## 1. 분석 질문과 외부 데이터 필요성

- **분석 질문:** 월별 `completed` 주문 금액과 월별 공휴일 수는 함께 변하는가?
- **내부 데이터로 알 수 있는 것:** `orders_clean.csv` + `order_items_clean.csv`의 월별 완료 주문 상세 금액 합계.
- **내부 데이터에 없는 것:** 대한민국 월별 공식 공휴일 수.
- **최소 외부 데이터:** 내부 주문과 겹치는 기간의 공식 공휴일 날짜와 공휴일 여부만 필요.
- **연결 단위:** `order_month` (YYYY-MM), 월별 1행으로 만든 후 병합.
- **외부 데이터 없이 가능한가?** 월별 완료 주문 매출 계산은 가능하지만 공휴일과의 관계는 답할 수 없음.

### 나의 판단
전체 검색결과·날씨·관광·SNS를 무작정 가져올 이유가 없습니다. 질문에 필요한 최소한의 공휴일 날짜 정보만 우선 검토합니다.

## 2. 공식 출처·이용조건 확인

**실행 후보:** 한국천문연구원 **특일 정보** (공공데이터포털)

- 공식 Source URL: https://www.data.go.kr/data/15012690/openapi.do
- 제공기관: 한국천문연구원
- 출처 유형: 공식 API (REST, **XML**)
- 공식 페이지 표시: 무료, 이용허락범위 제한 없음, 개발·운영 활용 자동승인, 개발계정 트래픽 10,000.
- 페이지에서 확인 가능한 기초 요청 항목: `ServiceKey`, `pageNo`, `numOfRows`, `solYear`, `solMonth`.
- 데이터 기준 기간: **아직 수집하지 않았으므로 미확정.** 실제 수집 시 내부 주문 범위와 교집합을 적용.
- 수집 시각 UTC: **실제 수집 없음.**
- 공휴일 세부 endpoint/response/XML path, 승인된 인증키와 적용되는 최신 조건: 실제 호출 직전 재확인.
- 개인정보: 공휴일 날짜 자체에는 개인 식별정보가 불필요.

**공식 문서 확인 시점:** 2026-10-14. 이는 **출처 웹페이지 검토일**이지 **API 데이터 수집일이 아닙니다.**

### 왜 이 출처인가?
공식 제공기관의 공휴일 조회 자료라서 블로그나 임의 달력보다 출처와 날짜 정의를 검증하기 좋습니다. 단, 서비스 등록정보만 확인했다는 이유로 개인 계정의 호출 승인·정확한 endpoint·응답 유효성까지 확인했다고 말할 수 없습니다.

### 해석 한계
한국 공식 공휴일이 가게의 영업일이나 실제 배송 중단일과 동일하다고 가정할 수 없으며, 대체공휴일·임시공휴일 적용 범위도 실제 수집 후 검증해야 합니다.

## 3. Network Gate 및 Secret 상태

공식 Runner가 만든 Evidence를 확인합니다. **RUN 플래그 네 개가 모두 False**인 것이 정상적인 시작입니다.

실제 Secret 내용은 출력하지 않습니다.

In [ ]:
from src.external_data_collection import (
    run_external_data_collection_setup,
    build_http_session,
    validate_external_dataframe,
    merge_external_data,
    sha256_file,
    versioned_snapshot_path,
)
setup = run_external_data_collection_setup(
    base_dir=ROOT,
    report_dir=REPORT_DIR,
)
out = setup["outputs"]
gate = out["network_gate"]
env_status = out["env_status"]
display(gate)
display(env_status)

assert gate["default"].eq(False).all()
assert env_status["value_exposed"].eq(False).all()

# Network 요청이 아니라 Session 정책 객체만 만듭니다.
session = build_http_session(total_retries=3, backoff_factor=0.5)
print("Network Gate: SAFE_DEFAULT")
print("Secret exposure: False")
print("HTTP requests made by this notebook: 0")

### 사람 판단

- `RUN_PUBLIC_API=False` / `RUN_NAVER_API=False` / `RUN_CRAWLING_EXAMPLE=False` / `POLICY_CONFIRMED=False` 유지.
- 출처·이용조건의 **공개 페이지는 확인**했지만 **개별 활용신청 승인/인증정보/최소 요청 범위와 세부 인터페이스 검증이 끝나지 않았으므로** 실제 요청은 실행하지 않습니다.
- Secret 상태는 `MISSING`·`PLACEHOLDER`·`CONFIGURED` 중 상태만 보고, 값 또는 일부 문자열도 노출하지 않습니다.
- **최종 Network Gate: HOLD**, 실제 수집 상태: **SKIPPED**.

네트워크 실행 전에는 공식 API 경로/인증/parameter/XML response/오류 구조, 호출 제한, 이용조건, 최소 기간, Snapshot 경로에 대한 확인과 명시적 승인이 필요합니다.

## 4. 요청 계약: timeout, retry, rate limit, pagination

현재 수업 공통 구현의 설정 예:
- Connect Timeout / Read Timeout: `(3.05, 20)`초.
- GET 재시도: 최대 3회, 지수 backoff 0.5, 일시 오류(429, 일부 5xx) 중심.
- **retry ≠ rate limit**: 재시도 정책만 설정해도 초당·일별 호출량 한도를 준수하는 것은 아닙니다.
- **HTTP 200 ≠ 업무 성공**: Content-Type, 응답 크기, XML/JSON 파싱, API 내부 `resultCode`, 페이지 종료 조건을 함께 검사해야 합니다.
- 실제 후보 API 포맷은 **XML**이므로 JSON 전용 호출 함수를 그대로 대입해서는 안 됩니다.
- XML response 세부 경로와 공휴일 전용 endpoint, rate limit 및 오류 문구는 호출 직전 공식 명세에서 확인합니다.
- 이번 Notebook은 인증키 전달·실제 URL 호출·rate limit 시험·pagination을 수행하지 않습니다.

## 5. Raw Snapshot / Metadata / SHA-256

실제 수집 후에는 `data/external/raw`, `data/external/processed`, `data/external/metadata`를 분리합니다.

- Raw: 응답 원본을 보존, timestamp 경로 생성, overwrite 금지.
- Processed: 타입/날짜/공휴일 플래그 검증 후 필요한 월별 1행으로 집계.
- Metadata: 제공기관, 공식 URL, 데이터 기준일, UTC 수집시각, 이용조건, 요청 범위, raw/processed 경로, SHA-256, 보유기간 및 재배포 여부 등을 기록.
- **수집 전 상태:** Raw 없음, Processed 외부 없음, Metadata 실측 없음, SHA-256 없음. **가짜 hash를 만들지 않습니다.**
- SHA-256은 같은 내용인지 검출할 뿐 정확성·최신성·대표성을 보증하지 않습니다.

In [ ]:
# 원본 저장 함수는 '기존 파일 보호' 정책만 코드로 확인합니다.
snapshot_dir = setup["paths"]["raw"]
metadata_dir = setup["paths"]["metadata"]
print("raw directory:", snapshot_dir)
print("metadata directory:", metadata_dir)
print("Versioned path policy: overwrite=False")
print("Actual raw snapshot count for this assignment: 0 (no data collected)")
print("Actual metadata SHA-256: NOT_AVAILABLE")

source_review = pd.DataFrame([{
    "provider": "한국천문연구원",
    "official_url": "https://www.data.go.kr/data/15012690/openapi.do",
    "source_type": "official_api_xml",
    "docs_review_date": "2026-10-14",
    "data_reference_period": "NOT_COLLECTED",
    "collected_at_utc": "",
    "license_note": "공공데이터포털: 이용허락범위 제한 없음 (실행 시 재확인)",
    "network_gate": "HOLD",
    "actual_collection_status": "SKIPPED",
    "raw_snapshot": "",
    "metadata_sha256": "",
    "api_execution_approved": False,
}])
source_review.to_csv(REPORT_DIR / "ch13_selected_source_review.csv", index=False, encoding="utf-8-sig")
display(source_review)

### 2·4단계 증빙 이미지 안내

이하 그림은 **공식 문서의 확인사항을 정리한 요약판과 실행 상태를 시각화한 이미지**입니다. **공식 웹페이지를 실제로 캡처한 스크린샷은 아닙니다.** 화면 캡처가 필요하면 공공데이터포털의 이용조건 영역에서 인증키·계정정보가 노출되지 않게 별도로 촬영할 수 있습니다.

In [ ]:
def evidence_card(filename, title, lines):
    fig, ax = plt.subplots(figsize=(10.2, 5.2))
    ax.axis("off")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.text(0.04, 0.91, title, fontsize=17, fontweight="bold", va="top")
    for i, line in enumerate(lines):
        ax.text(0.05, 0.77 - i*0.115, line, fontsize=12, va="top")
    fig.tight_layout()
    fig.savefig(IMAGE_DIR / filename, dpi=145, bbox_inches="tight")
    plt.show()

evidence_card("step02_source_terms.png", "Source & Terms — DOCUMENTED SUMMARY", [
    "Provider: Korea Astronomy and Space Science Institute (KASI)",
    "Catalog: data.go.kr/data/15012690/openapi.do",
    "Type: official API / REST / XML",
    "Portal: free / unrestricted use scope / application required",
    "Docs reviewed: 2026-10-14; API request: NOT EXECUTED",
    "This is a summary figure, NOT a website screenshot.",
])
evidence_card("step04_snapshot_metadata.png", "Raw / Processed / Metadata — NOT COLLECTED", [
    "Network gate: HOLD; actual request count: 0",
    "data/external/raw/: no acquired API payload",
    "data/external/processed/: no acquired API data",
    "data/external/metadata/: no collected snapshot metadata",
    "Real collected_at_utc: N/A    SHA-256: N/A",
    "On a future approved run: timestamp + no overwrite + hash.",
])

![공식 출처·이용조건 요약 — 웹페이지 캡처 아님](images/step02_source_terms.png)

![Raw / Metadata 미수집 상태](images/step04_snapshot_metadata.png)

## 6. 내부 주문 데이터 확인 및 월별 집계

이 단계는 저장소 `data/processed`를 읽습니다. 외부 데이터를 호출하는 것이 아닙니다.

- `completed` 주문만 포함.
- `line_total = quantity × unit_price`를 검산.
- 주문 상세는 주문당 여러 행이므로 `orders`는 `many_to_one` 관계로 결합.
- `order_month`별 완료 주문 상세 금액 총합을 계산.
- 주문 상세 합계와 월별 집계 총합의 일치 확인.

In [ ]:
orders = pd.read_csv(PROCESSED_DIR / "orders_clean.csv")
items = pd.read_csv(PROCESSED_DIR / "order_items_clean.csv")

required_orders = {"order_id", "order_status", "order_month"}
required_items = {"order_id", "quantity", "unit_price", "line_total"}
assert required_orders <= set(orders.columns)
assert required_items <= set(items.columns)
assert not orders["order_id"].duplicated().any()

for col in ("quantity", "unit_price", "line_total"):
    items[col] = pd.to_numeric(items[col], errors="raise")
line_mismatch = int(
    ((items["line_total"] - items["quantity"] * items["unit_price"]).abs() > 0.000001).sum()
)
assert line_mismatch == 0

completed_ids = orders.loc[orders["order_status"].eq("completed"), ["order_id", "order_month"]]
completed_items = items.merge(
    completed_ids, on="order_id", how="inner", validate="many_to_one"
)
internal_monthly = (
    completed_items.groupby("order_month", as_index=False)["line_total"]
    .sum()
    .rename(columns={"line_total": "completed_order_amount"})
    .sort_values("order_month").reset_index(drop=True)
)
internal_monthly.to_csv(REPORT_DIR / "ch13_internal_monthly_completed_sales.csv",
                        index=False, encoding="utf-8-sig")
source_total = float(completed_items["line_total"].sum())
group_total = float(internal_monthly["completed_order_amount"].sum())
assert abs(source_total - group_total) < 1e-6

print("Completed item rows:", len(completed_items))
print("Distinct months:", len(internal_monthly))
print("line_total mismatch:", line_mismatch)
print("Source/group total match:", abs(source_total - group_total) < 1e-6)
display(internal_monthly.head(8))

## 7. 외부 데이터 품질 검증: 실제 상태 NO_DATA

공휴일 API를 호출하지 않았으므로 **실제 외부 품질 검증은 실행하지 않았습니다**. 행 수·키 중복·날짜 파싱·월별 공휴일 수를 실제로 확인한 것처럼 기록하지 않습니다.

대신 아래 *명시적 합성 데모*에서 API 데이터가 들어왔을 때 사용할 공통 함수의 기본 계약만 테스트합니다.

실제 수집을 실행할 때는 공휴일 중복 날짜, `isHoliday` 정의, 응답 오류코드, 월별 키 누락/중복, 전체 요청 기간 등을 별도 확인해야 합니다.

In [ ]:
actual_external_quality = pd.DataFrame([{
    "scope": "actual_external_holiday_api",
    "collection_status": "SKIPPED",
    "quality_status": "NO_DATA",
    "rows": "NOT_AVAILABLE",
    "key_missing": "NOT_EVALUATED",
    "key_duplicates": "NOT_EVALUATED",
    "date_parse_failures": "NOT_EVALUATED",
    "reason": "Network Gate HOLD / no API request",
}])
actual_external_quality.to_csv(REPORT_DIR / "ch13_actual_external_quality_status.csv",
                               index=False, encoding="utf-8-sig")
display(actual_external_quality)

# 아래는 실제 외부 데이터가 아니라 병합 계약 확인을 위한 합성값입니다.
selected_months = internal_monthly["order_month"].head(3).tolist()
assert len(selected_months) >= 3
external_demo = pd.DataFrame({
    "order_month": selected_months,
    "synthetic_holiday_count": [1, 2, 3],
})
demo_quality = validate_external_dataframe(external_demo, key_columns="order_month")
demo_quality["data_origin"] = "SYNTHETIC_DEMO_ONLY"
demo_quality.to_csv(REPORT_DIR / "ch13_synthetic_quality_checks.csv",
                    index=False, encoding="utf-8-sig")
display(demo_quality)
assert not demo_quality["status"].eq("FAIL").any()

In [ ]:
evidence_card("step05_quality.png", "External Data Quality — REAL VS DEMO", [
    "REAL holiday API: SKIPPED / NO_DATA / not validated",
    "No raw payload, response rows, or parsed holiday records.",
    "SYNTHETIC demo: monthly key check only (3 fabricated counts)",
    "Demo key missing: 0; demo key duplicates: 0",
    "Synthetic values are NOT actual holiday counts.",
    "Real quality can be judged only after an approved collection.",
])

![실제 미수집 / 합성 검증 구분](images/step05_quality.png)

### 결과 관찰 및 판단

공통 검증 함수를 돌릴 수 있다는 것과 **공식 외부 데이터가 유효하다는 것**은 별개입니다. 현재 실제 데이터는 `NO_DATA`이므로 분석 사용 여부를 판단할 단계가 아닙니다.

## 8. 내부·외부 분석 단위 정렬

| 점검 | 내부 | 외부 목표 |
|---|---|---|
| 한 행 | 월별 completed 금액 | 월별 공휴일 수 |
| 시간 | 주문 생성 월, YYYY-MM | 실제 날짜 → YYYY-MM |
| 시간대 | 입력 날짜 기준 (KST 여부 검토) | 한국 공휴일 날짜 |
| 지역 | 내부 주문에 지역 정보 없음 | 대한민국 전체 공휴일 |
| Key | order_month | order_month |
| 기대 병합 | one_to_one | one_to_one |

실제 공휴일 목록은 날짜별 다수 행이므로 월별 집계를 선행하고, 오른쪽 `order_month`가 고유한지 검증한 뒤 병합합니다.

## 9. 병합 검증 — 합성 데모만 실행

오른쪽 키가 중복되면 행 수가 늘어나거나 잘못된 매출 수치를 만들 수 있어 fail-fast합니다. 외부 미매칭은 **실제 0과 다른 ‘데이터 없음’**이므로 자동으로 0을 채우지 않습니다.

현재는 공식 공휴일 데이터가 없어 실제 병합은 수행하지 않습니다.

In [ ]:
demo_internal = internal_monthly.head(3).copy()
merged_demo, merge_check_demo = merge_external_data(
    demo_internal, external_demo,
    on="order_month", how="left", validate="one_to_one",
)
merge_check_demo["data_origin"] = "SYNTHETIC_DEMO_ONLY"
merge_check_demo.to_csv(REPORT_DIR / "ch13_synthetic_merge_checks.csv",
                        index=False, encoding="utf-8-sig")
display(merged_demo)
display(merge_check_demo)
assert merge_check_demo["status"].eq("PASS").all()
assert merge_check_demo["right_key_duplicate_count"].eq(0).all()
assert merge_check_demo["row_count_preserved"].eq(True).all()

# 잘못된 오른쪽 중복 데이터는 실패해야 합니다.
duplicate_right = pd.concat([external_demo, external_demo.iloc[[0]]], ignore_index=True)
try:
    merge_external_data(demo_internal, duplicate_right, on="order_month", validate="one_to_one")
except ValueError:
    print("Duplicate right key → expected fail-fast: PASS")
else:
    raise AssertionError("오른쪽 중복키가 차단되지 않았습니다.")

In [ ]:
evidence_card("step07_merge.png", "Merge Contract — SYNTHETIC DEMO ONLY", [
    "Actual API data: NO_DATA; no actual merge was performed",
    "Internal (real) 3 months + synthetic (fabricated) 3 months",
    "join key: order_month; validate: one_to_one",
    "Right duplicate keys: 0; row count preserved: True",
    "Unmatched in demo: 0 (does NOT indicate real-data coverage)",
    "Deliberate duplicate right key: rejected as expected",
])

![합성 데이터 병합 검증 — 실제 결과 아님](images/step07_merge.png)

### 해석
이 통과 결과는 **병합 코드가 특정 합성 예제에서 키 제약을 지켰다는 Evidence**일 뿐 실제 공휴일과 주문 금액 사이의 관계를 보여주지 않습니다.

## 10. 검색 API 및 공개 HTML의 대표성·정책 한계

이번 과제에서는 검색 API와 HTML 크롤링을 수행하지 않습니다.

- 검색 Query/Sort/Page/Collected At UTC: **해당 없음**
- 검색결과는 검색 서비스의 색인·정렬·페이지·수집 시점에 제한됨. 전체 인터넷이나 시장의 대표 표본으로 해석 금지.
- 공식 파일/API 우선 검토. HTML이 꼭 필요할 때는 robots.txt, 이용약관, 라이선스, 개인정보, 요청량을 각각 확인.
- **robots.txt 허용 ≠ 이용허락 완료**
- 로그인/CAPTCHA/paywall/접근제한 우회 없음.
- 외부 웹·PDF의 지시문은 `untrusted data`로 취급. Secret 공개·OS 명령·파일 삭제 등 지시는 실행하지 않음.

## 11. 실행 기록: 실제 수집과 데모 명확히 구분

가이드의 실행 상태는 서로 다릅니다.

- 실제 API 데이터 수집: **SKIPPED** — Network Gate HOLD.
- 실제 외부 품질 검증: **NO_DATA**.
- 자체 생성한 3행 공휴일 수 유사 컬럼: **SYNTHETIC_DEMO_ONLY**.
- 실제 내부 데이터의 월별 매출 계산: **EXECUTED (internal data only)**.
- 미래 실제 승인 수집의 Raw/Metadata/Hash: **NOT_AVAILABLE**.

외부 데이터 수집이 없는 현재 상태에서 공휴일과 매출 간 통계적 관계나 인과관계를 주장하지 않습니다.

In [ ]:
decision = pd.DataFrame([
    {"scope":"actual_external_collection", "execution_status":"SKIPPED",
     "quality_status":"NO_DATA", "requests":0,
     "reason":"Network Gate HOLD, no approved API request"},
    {"scope":"external_monthly_join_demo", "execution_status":"SYNTHETIC_DEMO_ONLY",
     "quality_status":"DEMO_PASS", "requests":0,
     "reason":"Three clearly fabricated counts used solely for join validation"},
    {"scope":"internal_monthly_completed_sales", "execution_status":"EXECUTED",
     "quality_status":"PASS", "requests":0,
     "reason":"Computed from repository processed CSV files"},
])
decision.to_csv(REPORT_DIR / "ch13_student_execution_decision.csv",index=False,encoding="utf-8-sig")
display(decision)

manifest = [
  "ch13_external_data_plan.csv",
  "ch13_collection_method_summary.csv",
  "ch13_external_integration_plan.csv",
  "ch13_external_data_checklist.csv",
  "ch13_external_data_log.csv",
  "ch13_env_key_status.csv",
  "ch13_collection_metadata_template.csv",
  "ch13_api_code_review_checklist.csv",
  "ch13_network_execution_gate.csv",
  "ch13_external_data_summary.md",
  "ch13_selected_source_review.csv",
  "ch13_internal_monthly_completed_sales.csv",
  "ch13_actual_external_quality_status.csv",
  "ch13_synthetic_quality_checks.csv",
  "ch13_synthetic_merge_checks.csv",
  "ch13_student_execution_decision.csv",
]
missing_reports = [name for name in manifest if not (REPORT_DIR / name).is_file()]
assert not missing_reports, missing_reports
print("Generated report files:", len(manifest))
print("Source policy review: documented, run permission not verified")
print("Actual network calls:", 0)
print("Real external collection status: SKIPPED")

## 12. 최종 사용 판단

**판단: 현재 사용 보류 (외부 공휴일 변수를 사용한 실제 분석)**

### 판단 근거
1. 분석 질문·공식 제공기관·데이터 출처 및 공공데이터포털 이용조건은 확인했지만 실제 공휴일 데이터가 수집되지 않았습니다.
2. 인증·구체적 API 호출 승인 및 공휴일 endpoint/response 계약을 최종 검증하지 않았으므로 Gate가 OFF입니다.
3. 실제 Raw Snapshot, Metadata, SHA-256, 품질 및 병합 결과가 없어 외부 변수의 유효성이나 대표성을 평가할 수 없습니다.

### 사용 가능한 부분
- 내부 processed 데이터만 이용한 월별 completed 주문 상세 금액 집계.
- Safe default, Secret 비노출, Snapshot/Metadata 설계.
- 합성값을 이용한 기본 병합 코드 계약 테스트.

### 아직 말할 수 없는 것
- 실제 공휴일 수와 매출이 함께 움직이는지.
- 공식 공휴일이 실제 영업·배송 휴무일인지.
- 공휴일이 매출 변화를 **원인적으로 일으켰는지**.

### 추가 작업
1. 공공데이터포털에서 해당 API 활용신청 및 Key·공휴일 전용 호출 계약 확인.
2. 실제 요청 범위·페이지 종료조건·정책 승인 후 최소 범위 수집.
3. Timestamped Raw / Metadata / SHA-256 작성, 실제 품질 검사.
4. 날짜 → 월 집계, 기간 일치, 오른쪽 키·미매칭 검증 후 해석.

## 13. 제출 체크리스트

- [x] 외부 데이터 필요성과 최소 범위를 질문에서 정의함.
- [x] 공식 출처 및 포털 라이선스/비용/신청 조건 확인.
- [x] Data Reference Date와 실제 Collected At을 구분함.
- [x] Network Gate 기본 OFF 확인, 승인 없는 수집 없음.
- [x] Secret 값 출력·기록 없음.
- [x] Timeout / retry / rate limit / pagination 개념 구분.
- [x] HTTP 200과 API 업무 성공 구분.
- [x] Raw / Processed / Metadata 보존 구조 설계.
- [x] Raw 미덮어쓰기/Hash 원칙 기록.
- [x] 실제 품질 = NO_DATA, 합성 품질 = DEMO PASS 분리.
- [x] 내부·외부 시간 단위를 YYYY-MM으로 정렬하도록 설계.
- [x] 합성 오른쪽 키 중복·병합 행 수 확인.
- [x] 외부 검색 결과 대표성/인과/Prompt Injection 한계 기록.
- [x] 실제 미수집을 가짜 결과로 대체하지 않음.
- [x] 보고서 16개 / 노트북 / 이미지 4개 재현성 확인.

**최종 Notebook URL:** https://github.com/cyw0927/-llm-data-analysis-course/blob/main/assignments/chapter13/chapter13.ipynb

**주의:** PNG는 직접 실행 증빙/상태 요약 그림이며, `step02_source_terms.png`는 공식 페이지 실제 스크린샷이 아닙니다.